# Demo: objectness-guided mixed-resolution detection

This notebook runs a trained objectness-guided detector on one validation
image and shows each step:

1. load the model from its run directory,
2. load an image and its ground-truth boxes,
3. the objectness map predicted by the objectness scorer,
4. the mixed-resolution patches the quadtree selects under the token budget,
5. the detections.

It needs a trained adaptive objectness model (see *Training* in the README:
uniform 832 baseline → objectness scorer → adaptive model) and the dataset
under `$DATA_ROOT` (see `data/README.md`).

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from matplotlib.patches import Patch, Rectangle
from PIL import Image

from patch_where_objects_are.config.detection import load_detection_config
from patch_where_objects_are.data.detect_dataset import json_split
from patch_where_objects_are.data.transforms import DetectionTransformPipeline
from patch_where_objects_are.models.factory import (
    build_detector_from_config,
    checkpoint_allow_missing_prefixes,
)
from patch_where_objects_are.utils.checkpoint import load_checkpoint

OUTPUT_ROOT = Path(os.environ["OUTPUT_ROOT"])

# Run directory of the adaptive model (e.g. sea, lars or modd2).
RUN_DIR = OUTPUT_ROOT / "adaptive_objectness_sea_ft1_n576" / "s0"
FILE_NAME = None  # None: first annotated validation image
SCORE_THRESHOLD = 0.3  # minimum detection score to plot

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Load the model

Every run stores its resolved `config.yaml` next to the checkpoints. The config
builds the detector, including the objectness selector, which loads the frozen
objectness scorer from `backbone.scorer_checkpoint`.

In [ ]:
cfg = load_detection_config(str(RUN_DIR / "config.yaml"))
model = build_detector_from_config(cfg)
load_checkpoint(
    model,
    None,
    None,
    str(RUN_DIR / "checkpoint_latest.pth"),
    allow_missing_prefixes=checkpoint_allow_missing_prefixes(cfg),
)
model.to(device).eval()

IMG_SIZE = cfg.backbone.img_size  # square input resolution
P = cfg.backbone.patch_size  # finest patch size p
print(
    f"dataset: {cfg.data.dataset}, selector: {cfg.backbone.patch_selector}, "
    f"token budget: {cfg.backbone.token_budget}, "
    f"input: {IMG_SIZE}x{IMG_SIZE}, base grid: {IMG_SIZE // P}x{IMG_SIZE // P}"
)

## 2. Load an image

The validation annotations and image root come from `cfg.data`. The image is
resized to the square input resolution without preserving the aspect ratio. All
plots below are drawn on the original image.

In [ ]:
with open(cfg.data.val_json) as f:
    coco = json.load(f)
annotated_ids = {a["image_id"] for a in coco["annotations"]}
info = next(
    img
    for img in coco["images"]
    if (img["file_name"] == FILE_NAME if FILE_NAME else img["id"] in annotated_ids)
)
file_name = info["file_name"]
gt_boxes = [
    a["bbox"] for a in coco["annotations"] if a["image_id"] == info["id"]
]  # xywh
class_names = [c["name"] for c in sorted(coco["categories"], key=lambda c: c["id"])]

split_dir = Path(cfg.data.image_root) / json_split(cfg.data.val_json)
image = Image.open(split_dir / file_name).convert("RGB")
W, H = image.size
x, _ = DetectionTransformPipeline(img_size=IMG_SIZE, augment=False)(image)
x = x[None].to(device)  # (1, 3, IMG_SIZE, IMG_SIZE)
sx, sy = W / IMG_SIZE, H / IMG_SIZE  # input -> original pixel scale

GT_COLOR, PRED_COLOR = "#1baf7a", "#eb6834"


def draw_boxes(ax, boxes_xywh, color, **kw):
    for bx, by, bw, bh in boxes_xywh:
        ax.add_patch(
            Rectangle(
                (bx, by), bw, bh, fill=False, edgecolor=color, linewidth=1.5, **kw
            )
        )


fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(image)
draw_boxes(ax, gt_boxes, GT_COLOR)
ax.set_title(f"{file_name} ({W}x{H}), {len(gt_boxes)} ground-truth boxes")
ax.axis("off")
plt.show()

## 3. Objectness map

The objectness scorer predicts one objectness score per finest-level cell
(`p x p` pixels of the input), i.e. a finest-level score map on the base grid.

In [ ]:
with torch.no_grad():
    objectness = model.selector.stem(x)[0].sigmoid().cpu()  # (Hp, Wp)

blues = plt.matplotlib.colors.LinearSegmentedColormap.from_list(
    "blues", ["#cde2fb", "#0d366b"]
)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].imshow(image)
axes[0].set_title("Image")
im = axes[1].imshow(objectness, cmap=blues, vmin=0, vmax=1, extent=(0, W, H, 0))
axes[1].set_title(f"Objectness map ({objectness.shape[0]}x{objectness.shape[1]} cells)")
fig.colorbar(im, ax=axes[1], fraction=0.03, label="objectness")
for ax in axes:
    ax.axis("off")
plt.show()

## 4. Mixed-resolution patches

The budgeted quadtree starts from 64 px patches and repeatedly splits the patch
with the highest score (the max of the objectness scores of the finest-level
cells it covers) into four children, until the next split would exceed the token
budget. Regions with objects get fine 16 px patches; background stays at 32 px
and 64 px.

In [ ]:
with torch.no_grad():
    pf = model.selector(x)[
        0
    ]  # Patchification: scale m and top-left base cell per token

SCALE_COLORS = {1: "#2a78d6", 2: "#eb6834", 4: "#1baf7a"}  # patch size m * p
counts = {m: int((pf.scales == m).sum()) for m in SCALE_COLORS}

fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(image)
for m, (gy, gx) in zip(pf.scales.tolist(), pf.base_indices.tolist(), strict=True):
    ax.add_patch(
        Rectangle(
            (gx * P * sx, gy * P * sy),
            m * P * sx,
            m * P * sy,
            fill=False,
            edgecolor=SCALE_COLORS[m],
            linewidth=0.8,
        )
    )
ax.legend(
    handles=[
        Patch(edgecolor=c, fill=False, label=f"{m * P} px: {counts[m]} tokens")
        for m, c in SCALE_COLORS.items()
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, 0),
    ncol=3,
    frameon=False,
)
ax.set_title(
    f"{pf.num_tokens} tokens (uniform {P} px patches: {pf.grid_hw[0] * pf.grid_hw[1]})"
)
ax.axis("off")
plt.show()

## 5. Detections

The detector returns boxes in input pixels (xyxy), scores and class indices
(into the sorted dataset categories); the boxes are scaled back to the original
image.

In [ ]:
with torch.no_grad():
    det = model(x)[0]

keep = det["scores"] >= SCORE_THRESHOLD
boxes = det["boxes"][keep].cpu() * torch.tensor([sx, sy, sx, sy])
pred_boxes = [(x0, y0, x1 - x0, y1 - y0) for x0, y0, x1, y1 in boxes.tolist()]

fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(image)
draw_boxes(ax, gt_boxes, GT_COLOR, linestyle="--")
draw_boxes(ax, pred_boxes, PRED_COLOR)
ax.legend(
    handles=[
        Patch(edgecolor=GT_COLOR, fill=False, linestyle="--", label="ground truth"),
        Patch(
            edgecolor=PRED_COLOR,
            fill=False,
            label=f"detections (score >= {SCORE_THRESHOLD})",
        ),
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, 0),
    ncol=3,
    frameon=False,
)
ax.set_title(f"{len(pred_boxes)} detections")
ax.axis("off")
plt.show()

for (bx, by, bw, bh), s, c in zip(
    pred_boxes, det["scores"][keep].tolist(), det["labels"][keep].tolist(), strict=True
):
    print(
        f"{class_names[c]:<13} score {s:.2f}  "
        f"box xywh ({bx:.0f}, {by:.0f}, {bw:.0f}, {bh:.0f})"
    )